# 05 — RAG Setup (Level 2: Rule-Based Knowledge Selection)

Approach: simpan curated knowledge di Supabase tabel sederhana (tanpa embedding/vector). Retrieval pakai metadata filter berdasarkan ML feature output.

**Kenapa Level 2 (no embedding)**:
- No RAM-heavy embedding model (kernel safe)
- Predictable retrieval (rule-based)
- Lebih cepat (no embedding compute)
- Sufficient untuk knowledge base kecil (~10 docs)

**Pipeline**:
```
User input lat/lng
     ↓
Extract features (ML)
     ↓
Rule-based topic selection (n_offices_500m > 30 → CBD topic)
     ↓
Supabase metadata filter
     ↓
Groq LLM (Llama 3.3 70B) generate narrative
```

## Setup

In [1]:
import sys
import os
import warnings
from pathlib import Path
from dotenv import load_dotenv

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
warnings.filterwarnings('ignore')

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')

SUPABASE_URL = os.environ['SUPABASE_URL']
SUPABASE_KEY = os.environ['SUPABASE_KEY']
GROQ_API_KEY = os.environ['GROQ_API_KEY']

print(f"✓ Env loaded")
print(f"  Supabase: {SUPABASE_URL[:40]}...")

✓ Env loaded
  Supabase: https://yionhkzheewqyvphxuip.supabase.co...


In [2]:
from supabase import create_client
from groq import Groq

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

GROQ_MODEL = 'llama-3.3-70b-versatile'

print("✓ Clients ready")

✓ Clients ready


## 1. Supabase Schema Setup

Run SQL ini sekali di Supabase SQL Editor:

```sql
-- Drop existing (kalau ada)
drop table if exists documents cascade;

-- Tabel sederhana tanpa vector
create table documents (
    id text primary key,
    content text not null,
    metadata jsonb,
    created_at timestamp default now()
);

-- Index untuk metadata search
create index documents_metadata_idx on documents using gin(metadata);
```

**Note**: gak ada `vector` extension, gak ada `embedding` column, gak ada `match_documents()` function. Simple.

Verify:

In [3]:
try:
    result = supabase.table('documents').select('id', count='exact').limit(1).execute()
    print(f"✓ Connected. Existing docs: {result.count}")
except Exception as e:
    print(f"✗ {type(e).__name__}: {e}")
    print("  Pastikan tabel 'documents' udah dibikin via SQL Editor")

✓ Connected. Existing docs: 10


## 2. Curated Knowledge dengan Topic Tags

Setiap doc punya `metadata.topic` untuk filter retrieval.

In [4]:
curated_content = [
    # Always-included general
    {
        'id': 'general_trends_2024',
        'content': '''Tren coffee shop Indonesia 2024 menunjukkan pertumbuhan signifikan di Jakarta. Konsumen Gen Z dan Millennial dominate (>60%). Specialty coffee dengan biji single origin Indonesia (Gayo, Toraja, Java Preanger) jadi pilihan segmen menengah-atas. Brand lokal seperti Kopi Kenangan, Janji Jiwa, Fore Coffee, Tomoro Coffee ekspansi agresif dengan model grab-and-go. Area CBD padat kompetisi, area pinggiran masih punya opportunity neighborhood cafe.''',
        'metadata': {'topic': 'general_trends', 'priority': 'high'}
    },
    
    # CBD/Office heavy area
    {
        'id': 'cbd_strategy',
        'content': '''Coffee shop sukses di area CBD/perkantoran padat (>30 kantor radius 500m) umumnya mengandalkan: (1) Speed of service — grab-and-go untuk commuter pagi 7-9 AM dan lunch break 12-2 PM, (2) Wifi stabil untuk meeting/coworking, (3) Harga menengah Rp 30-50rb, (4) Lokasi visible dari lobby gedung atau MRT/halte. Risk utama: high rent (Rp 1.5-3 jt/m²/bulan di SCBD), kompetisi sangat ketat. Cafe tanpa konsep diferensiasi sering tutup 1-2 tahun.''',
        'metadata': {'topic': 'cbd_strategy'}
    },
    
    # Residential area
    {
        'id': 'residential_strategy',
        'content': '''Coffee shop di area residential (kantor sedikit, populasi warga padat) sukses dengan konsep neighborhood-style: lebih spacious, atmosphere hangat untuk hangout/study. Target: warga setempat, mahasiswa, freelancer. Operating hours penting — buka sampai malam untuk weekend traffic. Harga lebih kompetitif (Rp 25-40rb). Lokasi dekat kompleks perumahan atau kampus jadi prime. Contoh sukses: Kemang, Tebet, Kebayoran.''',
        'metadata': {'topic': 'residential_strategy'}
    },
    
    # High competition
    {
        'id': 'high_competition_strategy',
        'content': '''Area dengan kompetisi tinggi (>15 cafe radius 500m) butuh diferensiasi kuat. Jangan compete on price — sulit untung. Pilihan diferensiasi: (1) Signature menu unik (es kopi susu Tuku style, signature drink branded), (2) Branding distinctive (visual identity strong, social media-ready), (3) Konsep niche (specialty grade A, plant-based, eco-friendly, work-friendly dengan power outlet banyak). Strategi opening: event marketing dengan influencer untuk generate buzz, lalu maintain quality consistent.''',
        'metadata': {'topic': 'high_competition'}
    },
    
    # Low competition
    {
        'id': 'low_competition_opportunity',
        'content': '''Area dengan kompetitor sedikit (<5 cafe radius 1km) bisa mean dua hal: (1) Underserved opportunity — area berkembang dengan demand tidak terpenuhi (Greenfield area, kompleks baru, area niche), atau (2) Low demand — emang gak ada market. Validasi: cek signal demand (populasi, kantor, sekolah, transit). Kalau ketiga signal kuat tapi cafe sedikit, ini opportunity bagus. Kalau lemah semua, hindari. First-mover advantage tinggi kalau market emerging.''',
        'metadata': {'topic': 'low_competition_opportunity'}
    },
    
    # Transit
    {
        'id': 'transit_advantage',
        'content': '''Cafe dekat MRT/halte busway (radius 500m) menikmati commuter traffic konsisten weekday. Boost morning traffic 30-50% vs cafe non-transit. Optimal positioning: visible dari pintu keluar transit + walking distance < 200m. Stasiun MRT prime: Sudirman, Bundaran HI, Senayan, Blok M, Lebak Bulus, Cipete, Fatmawati. Halte busway impactful di koridor 1 dan 4. KRL/LRT effect lebih kecil karena demografi penumpang berbeda.''',
        'metadata': {'topic': 'transit_advantage'}
    },
    
    # Mall anchor
    {
        'id': 'mall_anchor_strategy',
        'content': '''Cafe di area dengan mall anchor (radius 2km) dapat boost traffic shopping crowd. Mall = anchor pull untuk weekend + lunch crowd. Strategi: posisi streetside dengan visibility ke jalur menuju mall, atau dalam mall langsung. Trade-off rent: mall Rp 1.5-3 jt/m²/bulan, streetside Rp 500rb-1.2 jt/m²/bulan. Streetside dengan parking memadai sering outperform mall cafe untuk segmen non-impulse buyers.''',
        'metadata': {'topic': 'mall_anchor'}
    },
    
    # Cannibalization
    {
        'id': 'cannibalization_warning',
        'content': '''Membuka cabang terlalu dekat dengan store sendiri (<500m) berisiko kanibalisasi customer. Rule of thumb F&B Indonesia: minimum 800m-1km antar cabang same brand di area urban padat, atau 2-3km di suburban. Pengecualian: konsep berbeda (flagship vs kiosk vs drive-thru) bisa lebih dekat karena target customer berbeda. Selalu cek brand sendiri terdekat sebelum buka lokasi baru — kalau < 500m, butuh justifikasi konsep yang clear.''',
        'metadata': {'topic': 'cannibalization'}
    },
    
    # Rating interpretation
    {
        'id': 'rating_interpretation',
        'content': '''Interpretasi rating kompetitor sekitar untuk strategi entry: Rating avg > 4.5 = kompetitor punya quality + customer loyalty tinggi, sulit ambil customer mereka tanpa quality lebih baik. Rating avg 4.0-4.4 = kompetitor decent tapi ada room improvement, opportunity untuk cafe berkualitas tinggi. Rating avg < 4.0 = kompetitor lemah, opportunity besar untuk cafe quality. Total review >10000 di radius 1km = area aktif F&B dengan customer base mature.''',
        'metadata': {'topic': 'rating_analysis'}
    },
    
    # Failure patterns
    {
        'id': 'failure_patterns',
        'content': '''Pola gagal coffee shop di Jakarta: (1) Lokasi sepi dengan signage minim — customer gak tau ada cafe; (2) Pricing terlalu premium di area residential dengan ekspektasi murah; (3) Menu terlalu eksperimental tanpa signature recognizable; (4) Operating hours mismatch dengan area; (5) Tidak ada online presence/delivery integration; (6) Quality kopi inkonsisten karena rotasi staf cepat. Cafe yang skip delivery (GoFood, GrabFood) kalah compete dengan mainstream brand.''',
        'metadata': {'topic': 'failure_patterns', 'priority': 'high'}
    },
]

print(f"Total curated content: {len(curated_content)} docs")
print(f"Topics: {set(d['metadata']['topic'] for d in curated_content)}")

Total curated content: 10 docs
Topics: {'cannibalization', 'low_competition_opportunity', 'general_trends', 'cbd_strategy', 'rating_analysis', 'failure_patterns', 'mall_anchor', 'residential_strategy', 'high_competition', 'transit_advantage'}


## 3. Upsert ke Supabase

In [5]:
# Upsert (insert atau update kalau ID udah ada)
BATCH = 50
for i in range(0, len(curated_content), BATCH):
    batch = curated_content[i:i+BATCH]
    supabase.table('documents').upsert(batch).execute()
    print(f"  ✓ batch {i//BATCH + 1}: {len(batch)} docs")

# Verify
result = supabase.table('documents').select('id', count='exact').limit(1).execute()
print(f"\n✓ Total docs di Supabase: {result.count}")

  ✓ batch 1: 10 docs

✓ Total docs di Supabase: 10


## 4. Rule-Based Topic Selection

In [6]:
def select_relevant_topics(ml_data):
    """Pilih topic relevant berdasarkan kondisi lokasi (ML feature)."""
    topics = ['general_trends', 'failure_patterns']  # always include
    
    # Office density
    n_off = ml_data.get('n_offices_500m', 0)
    if n_off > 30:
        topics.append('cbd_strategy')
    elif n_off < 5:
        topics.append('residential_strategy')
    
    # Competition density
    n_comp = ml_data.get('n_competitors_500m', 0)
    if n_comp > 15:
        topics.append('high_competition')
    elif n_comp < 3:
        topics.append('low_competition_opportunity')
    
    # Transit
    if ml_data.get('n_transit_500m', 0) > 3:
        topics.append('transit_advantage')
    
    # Mall
    if ml_data.get('n_malls_2km', 0) > 0:
        topics.append('mall_anchor')
    
    # Cannibalization
    if ml_data.get('nearest_owner_store_m', 9999) < 1000:
        topics.append('cannibalization')
    
    # Rating context
    avg_rating = ml_data.get('avg_competitor_rating_2km', 0)
    if avg_rating > 0:
        topics.append('rating_analysis')
    
    return list(set(topics))  # dedupe


# Test
sample_ml = {
    'n_offices_500m': 45,
    'n_competitors_500m': 18,
    'n_transit_500m': 5,
    'n_malls_2km': 2,
    'nearest_owner_store_m': 1200,
    'avg_competitor_rating_2km': 4.3,
}
print(f"Selected topics for sample (CBD-like):")
print(select_relevant_topics(sample_ml))

Selected topics for sample (CBD-like):
['general_trends', 'cbd_strategy', 'rating_analysis', 'failure_patterns', 'mall_anchor', 'high_competition', 'transit_advantage']


## 5. Retrieval Function

In [7]:
def retrieve_context(ml_data):
    """Fetch curated docs dari Supabase based on topics yang relevant."""
    topics = select_relevant_topics(ml_data)
    
    # Query: ambil docs dengan topic in list
    result = supabase.table('documents').select('*').in_(
        'metadata->>topic', topics
    ).execute()
    
    return result.data, topics


# Test
docs, topics = retrieve_context(sample_ml)
print(f"Topics queried: {topics}")
print(f"Docs retrieved: {len(docs)}")
for d in docs[:3]:
    print(f"\n[{d['metadata'].get('topic')}]")
    print(d['content'][:150] + '...')

Topics queried: ['general_trends', 'cbd_strategy', 'rating_analysis', 'failure_patterns', 'mall_anchor', 'high_competition', 'transit_advantage']
Docs retrieved: 7

[general_trends]
Tren coffee shop Indonesia 2024 menunjukkan pertumbuhan signifikan di Jakarta. Konsumen Gen Z dan Millennial dominate (>60%). Specialty coffee dengan ...

[cbd_strategy]
Coffee shop sukses di area CBD/perkantoran padat (>30 kantor radius 500m) umumnya mengandalkan: (1) Speed of service — grab-and-go untuk commuter pagi...

[high_competition]
Area dengan kompetisi tinggi (>15 cafe radius 500m) butuh diferensiasi kuat. Jangan compete on price — sulit untung. Pilihan diferensiasi: (1) Signatu...


## 6. Prompt Template (Location-Focused)

In [8]:
PROMPT_TEMPLATE = '''Lo adalah analis lokasi F&B Indonesia yang berpengalaman. 
Berdasarkan data lokasi spesifik berikut, buat analisis kelayakan pembukaan coffee shop.

KOORDINAT KANDIDAT: ({lat}, {lng})

=== HASIL ANALISIS RADIUS ===

🏪 KOMPETISI:
- Cafe radius 500m: {n_competitors_500m} (rata-rata rating {avg_rating_500}, total {total_reviews_500} review)
- Cafe radius 2km: {n_competitors_2km} (rata-rata rating {avg_rating_2km}, total {total_reviews_2km} review)
- Cafe terpopuler radius 2km punya {max_reviews_2km} review (market leader signal)

🏠 STORE OWNER:
- Jarak store owner terdekat: {nearest_owner_store_m} meter
- {cannibalization_note}

🏢 SINYAL DEMAND:
- Kantor radius 500m: {n_offices_500m} ({office_signal})
- Kantor radius 2km: {n_offices_2km}
- Mall radius 2km: {n_malls_2km}
- Halte/transit radius 500m: {n_transit_500m} ({transit_signal})
- Sekolah radius 1km: {n_schools_1km}

=== HASIL ML MODEL ===
- Predicted score: {ml_score}/100
- Top 3 faktor pendorong: {shap_top3}

=== KONTEKS RELEVAN (knowledge base) ===
{retrieved_context}

=== TUGAS ===
Berdasarkan SEMUA data di atas, tulis analisis lokasi ini dalam Bahasa Indonesia, struktur:

1. **Ringkasan Eksekutif** (2-3 kalimat tegas tentang lokasi ini)
2. **Kekuatan Lokasi** (3-5 bullet, reference angka konkret dari data)
3. **Risiko & Kelemahan** (2-4 bullet dengan angka)
4. **Rekomendasi**: GO / HOLD / NO-GO + alasan utama (1 paragraf)
5. **Saran Tindak Lanjut** (3-5 action konkret untuk owner)

PENTING:
- Selalu reference angka konkret (e.g., "kompetitor 18 cafe dalam 500m")
- Fokus pada karakteristik lokasi dari data radius, bukan label administratif
- Tone profesional tapi accessible
'''

## 7. Generate Function

In [9]:
def build_cannibalization_note(distance):
    if distance < 500:
        return f"⚠️ RISIKO KANIBALISASI — terlalu dekat dengan store sendiri ({distance:.0f}m)"
    elif distance < 1000:
        return f"⚠️ Perlu hati-hati, store sendiri masih dekat ({distance:.0f}m)"
    elif distance < 3000:
        return f"OK — store sendiri di jarak aman ({distance:.0f}m)"
    else:
        return f"Tidak ada masalah kanibalisasi ({distance:.0f}m)"


def interpret_office(n):
    if n > 30: return "CBD/perkantoran padat"
    if n > 10: return "perkantoran moderat"
    if n > 0:  return "perkantoran sedikit"
    return "tidak ada perkantoran terdekat"


def interpret_transit(n):
    if n > 5: return "aksesibilitas transit sangat baik"
    if n > 0: return "ada akses transit"
    return "tidak ada transit terdekat"


def generate_summary(lat, lng, ml_data, retrieved_docs):
    nearest = ml_data.get('nearest_owner_store_m', 9999)
    n_off = ml_data.get('n_offices_500m', 0)
    n_tr = ml_data.get('n_transit_500m', 0)
    
    context = '\n\n'.join([f"- {d['content']}" for d in retrieved_docs])
    
    prompt = PROMPT_TEMPLATE.format(
        lat=lat, lng=lng,
        n_competitors_500m=ml_data.get('n_competitors_500m', 0),
        avg_rating_500=round(ml_data.get('avg_competitor_rating_500', 0), 2),
        total_reviews_500=int(ml_data.get('total_competitor_reviews_500', 0)),
        n_competitors_2km=ml_data.get('n_competitors_2km', 0),
        avg_rating_2km=round(ml_data.get('avg_competitor_rating_2km', 0), 2),
        total_reviews_2km=int(ml_data.get('total_competitor_reviews_2km', 0)),
        max_reviews_2km=int(ml_data.get('max_competitor_reviews_2km', 0)),
        nearest_owner_store_m=int(nearest),
        cannibalization_note=build_cannibalization_note(nearest),
        n_offices_500m=n_off,
        office_signal=interpret_office(n_off),
        n_offices_2km=ml_data.get('n_offices_2km', 0),
        n_malls_2km=ml_data.get('n_malls_2km', 0),
        n_transit_500m=n_tr,
        transit_signal=interpret_transit(n_tr),
        n_schools_1km=ml_data.get('n_schools_1km', 0),
        ml_score=ml_data.get('score', 0),
        shap_top3=', '.join(ml_data.get('top_factors', [])),
        retrieved_context=context,
    )
    
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=1200,
        temperature=0.2,
    )
    return response.choices[0].message.content

### Test End-to-End

In [10]:
# Load model + feature extractor untuk integration test
import joblib
import pandas as pd
import numpy as np
import math

model_artifact = joblib.load(ROOT / 'models' / 'xgb_demand.pkl')
model = model_artifact['model']
feature_cols = model_artifact['feature_cols']

# Load source data
cafes = pd.read_csv(ROOT / 'data' / 'clean' / 'cafes_clean.csv')
owner = pd.read_csv(ROOT / 'data' / 'clean' / 'owner_clean.csv')
poi_data = {}
for cat in ['office', 'mall', 'transit', 'school']:
    p = ROOT / 'data' / 'clean' / f'poi_{cat}_clean.csv'
    if p.exists():
        poi_data[cat] = pd.read_csv(p)

print(f"✓ Model + data loaded")

✓ Model + data loaded


In [11]:
def haversine_vec(lat1, lng1, lats, lngs):
    R = 6371000
    p1 = math.radians(lat1)
    p2 = np.radians(lats)
    dp = np.radians(lats - lat1)
    dl = np.radians(lngs - lng1)
    a = np.sin(dp/2)**2 + math.cos(p1) * np.cos(p2) * np.sin(dl/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def count_in_radius(lat, lng, df, radius_m):
    if df.empty: return 0
    d = haversine_vec(lat, lng, df['lat'].values, df['lng'].values)
    return int((d < radius_m).sum())


def stats_competitors(lat, lng, df, radius_m):
    d = haversine_vec(lat, lng, df['lat'].values, df['lng'].values)
    mask = d < radius_m
    if not mask.any():
        return {'n': 0, 'avg_rating': 0, 'total_reviews': 0, 'max_reviews': 0}
    subset = df[mask]
    return {
        'n': int(mask.sum()),
        'avg_rating': float(subset['rating'].mean()),
        'total_reviews': float(subset['reviews_count'].sum()),
        'max_reviews': float(subset['reviews_count'].max()),
    }


def extract_features_for_location(lat, lng):
    s500 = stats_competitors(lat, lng, cafes, 500)
    s2km = stats_competitors(lat, lng, cafes, 2000)
    
    d_owner = haversine_vec(lat, lng, owner['lat'].values, owner['lng'].values)
    nearest_owner = float(d_owner.min())
    
    feats = {
        'n_competitors_500m': s500['n'],
        'avg_competitor_rating_500': s500['avg_rating'],
        'total_competitor_reviews_500': s500['total_reviews'],
        'max_competitor_reviews_500': s500['max_reviews'],
        'n_competitors_2km': s2km['n'],
        'avg_competitor_rating_2km': s2km['avg_rating'],
        'total_competitor_reviews_2km': s2km['total_reviews'],
        'max_competitor_reviews_2km': s2km['max_reviews'],
        'nearest_owner_store_m': nearest_owner,
        'n_offices_500m': count_in_radius(lat, lng, poi_data.get('office', pd.DataFrame()), 500),
        'n_offices_2km': count_in_radius(lat, lng, poi_data.get('office', pd.DataFrame()), 2000),
        'n_malls_500m': count_in_radius(lat, lng, poi_data.get('mall', pd.DataFrame()), 500),
        'n_malls_2km': count_in_radius(lat, lng, poi_data.get('mall', pd.DataFrame()), 2000),
        'n_transit_500m': count_in_radius(lat, lng, poi_data.get('transit', pd.DataFrame()), 500),
        'n_transit_2km': count_in_radius(lat, lng, poi_data.get('transit', pd.DataFrame()), 2000),
        'n_schools_500m': count_in_radius(lat, lng, poi_data.get('school', pd.DataFrame()), 500),
        'n_schools_1km': count_in_radius(lat, lng, poi_data.get('school', pd.DataFrame()), 1000),
        'n_schools_2km': count_in_radius(lat, lng, poi_data.get('school', pd.DataFrame()), 2000),
    }
    
    # Derivative features (sesuaikan dengan yang dipakai saat training)
    feats['density_ratio_500m_2km'] = feats['n_competitors_500m'] / (feats['n_competitors_2km'] + 1)
    feats['avg_reviews_per_cafe_2km'] = feats['total_competitor_reviews_2km'] / (feats['n_competitors_2km'] + 1)
    feats['market_saturation'] = feats['n_competitors_500m'] * feats['avg_competitor_rating_2km']
    feats['office_transit_combo'] = feats['n_offices_500m'] * feats['n_transit_500m']
    feats['has_strong_competitor'] = int(feats['max_competitor_reviews_2km'] > 1000)
    feats['anchor_score'] = feats['n_malls_2km'] * 0.5 + feats['n_transit_500m'] * 1.0
    
    return feats


def predict_score(lat, lng):
    feats = extract_features_for_location(lat, lng)
    
    X = pd.DataFrame([{c: feats.get(c, 0) for c in feature_cols}])
    pred = float(model.predict(X)[0])
    score = min(100, max(0, pred * 2.5))
    
    import shap
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X)[0]
    top3 = [f for f, _ in sorted(zip(feature_cols, shap_vals),
                                  key=lambda x: abs(x[1]), reverse=True)[:3]]
    
    return {
        'score': round(score, 1),
        'top_factors': top3,
        **feats,
    }

In [12]:
def full_analysis(lat, lng):
    """End-to-end pipeline."""
    print(f"📍 Analyzing ({lat}, {lng})...")
    
    # 1. ML prediction
    ml_data = predict_score(lat, lng)
    print(f"  ✓ ML score: {ml_data['score']}/100")
    
    # 2. Retrieve relevant context
    docs, topics = retrieve_context(ml_data)
    print(f"  ✓ Retrieved {len(docs)} docs (topics: {topics})")
    
    # 3. Generate summary
    print(f"  → Generating narrative...")
    summary = generate_summary(lat, lng, ml_data, docs)
    
    return {
        'lat': lat, 'lng': lng,
        'ml_data': ml_data,
        'topics': topics,
        'summary': summary,
    }

In [13]:
# Test 1: Senopati (CBD high-density)
result = full_analysis(-6.2297, 106.8195)
print("\n" + "="*70)
print("NARRATIVE:")
print("="*70)
print(result['summary'])

📍 Analyzing (-6.2297, 106.8195)...
  ✓ ML score: 61.5/100
  ✓ Retrieved 6 docs (topics: ['cannibalization', 'low_competition_opportunity', 'general_trends', 'rating_analysis', 'failure_patterns', 'residential_strategy'])
  → Generating narrative...

NARRATIVE:
**Ringkasan Eksekutif**
Lokasi dengan koordinat (-6.2297, 106.8195) menunjukkan potensi yang rendah untuk pembukaan coffee shop baru karena kurangnya sinyal demand dan risiko kanibalisasi dengan store owner yang sama. Dengan skor prediksi 61,5/100, lokasi ini memerlukan pertimbangan yang hati-hati sebelum memutuskan untuk membuka coffee shop.

**Kekuatan Lokasi**
* Tidak ada kompetitor cafe dalam radius 500m, yang berarti tidak ada persaingan langsung.
* Lokasi ini memiliki potensi untuk menjadi pemimpin pasar karena tidak ada cafe lain yang dominan dalam radius 2km.
* Skor prediksi 61,5/100 menunjukkan bahwa lokasi ini memiliki beberapa kekuatan yang dapat dimanfaatkan.

**Risiko & Kelemahan**
* Risiko kanibalisasi dengan store 

In [14]:
# Test 2: Tebet (residential)
result = full_analysis(-6.2298, 106.8546)
print("\n" + "="*70)
print("NARRATIVE:")
print("="*70)
print(result['summary'])

📍 Analyzing (-6.2298, 106.8546)...
  ✓ ML score: 58.8/100
  ✓ Retrieved 6 docs (topics: ['cannibalization', 'general_trends', 'rating_analysis', 'failure_patterns', 'residential_strategy', 'high_competition'])
  → Generating narrative...

NARRATIVE:
**Ringkasan Eksekutif**
Lokasi dengan koordinat (-6.2298, 106.8546) memiliki tingkat kompetisi yang tinggi dengan 20 cafe dalam radius 500m dan 60 cafe dalam radius 2km. Namun, lokasi ini juga memiliki potensi besar karena rating rata-rata kompetitor yang tinggi, yaitu 4.55 dalam radius 500m. Dengan demikian, lokasi ini memerlukan strategi yang tepat untuk bersaing dengan kompetitor yang sudah ada.

**Kekuatan Lokasi**
* Lokasi ini memiliki kompetitor yang kuat dengan rating rata-rata 4.55 dalam radius 500m, menunjukkan bahwa konsumen di area ini memiliki selera yang tinggi dan siap membayar untuk kualitas yang baik.
* Jumlah review yang besar, yaitu 25.481 review dalam radius 500m, menunjukkan bahwa area ini memiliki trafik yang tinggi dan

In [15]:
# Test 3: Penjaringan (low-density)
result = full_analysis(-6.1255, 106.7700)
print("\n" + "="*70)
print("NARRATIVE:")
print("="*70)
print(result['summary'])

📍 Analyzing (-6.1255, 106.77)...
  ✓ ML score: 64.9/100
  ✓ Retrieved 5 docs (topics: ['low_competition_opportunity', 'general_trends', 'rating_analysis', 'failure_patterns', 'residential_strategy'])
  → Generating narrative...

NARRATIVE:
**1. Ringkasan Eksekutif**
Lokasi dengan koordinat (-6.1255, 106.77) menunjukkan potensi yang relatif rendah untuk pembukaan coffee shop, dengan hanya 0 kompetitor dalam radius 500m dan 17 kompetitor dalam radius 2km. Skor prediksi dari model ML adalah 64,9/100, menunjukkan bahwa lokasi ini memiliki beberapa kekuatan, tetapi juga beberapa kelemahan.

**2. Kekuatan Lokasi**
* Tidak ada kompetitor dalam radius 500m, menunjukkan bahwa lokasi ini memiliki potensi untuk menjadi pemimpin pasar di daerah tersebut.
* Jarak store owner terdekat sebesar 3881 meter, menunjukkan bahwa tidak ada masalah kanibalisasi yang signifikan.
* Skor prediksi dari model ML menunjukkan bahwa lokasi ini memiliki beberapa kekuatan, seperti potensi demand yang relatif tinggi.



## 9. Save Module untuk Streamlit

In [ ]:
retrieve_py = '''"""Knowledge retrieval — rule-based selection dari Supabase."""

import os
from supabase import create_client
from dotenv import load_dotenv

load_dotenv()

_supabase = create_client(
    os.environ['SUPABASE_URL'],
    os.environ['SUPABASE_KEY']
)


def select_relevant_topics(ml_data):
    """Pilih topic dari ML feature output."""
    topics = ['general_trends', 'failure_patterns']
    
    n_off = ml_data.get('n_offices_500m', 0)
    if n_off > 30:
        topics.append('cbd_strategy')
    elif n_off < 5:
        topics.append('residential_strategy')
    
    n_comp = ml_data.get('n_competitors_500m', 0)
    if n_comp > 15:
        topics.append('high_competition')
    elif n_comp < 3:
        topics.append('low_competition_opportunity')
    
    if ml_data.get('n_transit_500m', 0) > 3:
        topics.append('transit_advantage')
    if ml_data.get('n_malls_2km', 0) > 0:
        topics.append('mall_anchor')
    if ml_data.get('nearest_owner_store_m', 9999) < 1000:
        topics.append('cannibalization')
    if ml_data.get('avg_competitor_rating_2km', 0) > 0:
        topics.append('rating_analysis')
    
    return list(set(topics))


def retrieve_context(ml_data):
    """Fetch docs from Supabase by topic."""
    topics = select_relevant_topics(ml_data)
    result = _supabase.table('documents').select('*').in_(
        'metadata->>topic', topics
    ).execute()
    return result.data
'''

(ROOT / 'src' / 'rag' / 'retrieve.py').parent.mkdir(parents=True, exist_ok=True)
(ROOT / 'src' / 'rag' / 'retrieve.py').write_text(retrieve_py)
print('✓ Saved src/rag/retrieve.py')

In [ ]:
generate_py = '''"""LLM generation dengan Groq Llama 3.3 70B."""

import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

_client = Groq(api_key=os.environ['GROQ_API_KEY'])
GROQ_MODEL = 'llama-3.3-70b-versatile'

PROMPT_TEMPLATE = """Lo adalah analis lokasi F&B Indonesia yang berpengalaman.
Berdasarkan data lokasi berikut, buat analisis kelayakan pembukaan coffee shop.

KOORDINAT: ({lat}, {lng})

=== RADIUS ANALYSIS ===
Cafe 500m: {n_competitors_500m} (avg rating {avg_rating_500}, {total_reviews_500} review)
Cafe 2km: {n_competitors_2km} (avg rating {avg_rating_2km}, {total_reviews_2km} review)
Market leader 2km: {max_reviews_2km} review

Owner store terdekat: {nearest_owner_store_m}m — {cannibalization_note}

Kantor 500m: {n_offices_500m} ({office_signal})
Mall 2km: {n_malls_2km}
Transit 500m: {n_transit_500m} ({transit_signal})
Sekolah 1km: {n_schools_1km}

=== ML MODEL ===
Score: {ml_score}/100
Top faktor: {shap_top3}

=== KONTEKS ===
{retrieved_context}

=== TUGAS ===
Tulis dalam Bahasa Indonesia, struktur:
1. Ringkasan Eksekutif (2-3 kalimat)
2. Kekuatan Lokasi (3-5 bullet, sebutkan angka konkret)
3. Risiko & Kelemahan (2-4 bullet)
4. Rekomendasi: GO / HOLD / NO-GO + alasan
5. Saran Tindak Lanjut (3-5 action)
"""


def _cannibalization_note(d):
    if d < 500: return f"RISIKO KANIBALISASI ({d:.0f}m)"
    if d < 1000: return f"masih dekat ({d:.0f}m)"
    if d < 3000: return f"jarak aman ({d:.0f}m)"
    return f"tidak masalah ({d:.0f}m)"

def _office_signal(n):
    if n > 30: return "CBD padat"
    if n > 10: return "perkantoran moderat"
    if n > 0: return "perkantoran sedikit"
    return "tidak ada perkantoran"

def _transit_signal(n):
    if n > 5: return "transit sangat baik"
    if n > 0: return "ada transit"
    return "tidak ada transit"


def generate_summary(lat, lng, ml_data, retrieved_docs):
    nearest = ml_data.get('nearest_owner_store_m', 9999)
    
    prompt = PROMPT_TEMPLATE.format(
        lat=lat, lng=lng,
        n_competitors_500m=ml_data.get('n_competitors_500m', 0),
        avg_rating_500=round(ml_data.get('avg_competitor_rating_500', 0), 2),
        total_reviews_500=int(ml_data.get('total_competitor_reviews_500', 0)),
        n_competitors_2km=ml_data.get('n_competitors_2km', 0),
        avg_rating_2km=round(ml_data.get('avg_competitor_rating_2km', 0), 2),
        total_reviews_2km=int(ml_data.get('total_competitor_reviews_2km', 0)),
        max_reviews_2km=int(ml_data.get('max_competitor_reviews_2km', 0)),
        nearest_owner_store_m=int(nearest),
        cannibalization_note=_cannibalization_note(nearest),
        n_offices_500m=ml_data.get('n_offices_500m', 0),
        office_signal=_office_signal(ml_data.get('n_offices_500m', 0)),
        n_malls_2km=ml_data.get('n_malls_2km', 0),
        n_transit_500m=ml_data.get('n_transit_500m', 0),
        transit_signal=_transit_signal(ml_data.get('n_transit_500m', 0)),
        n_schools_1km=ml_data.get('n_schools_1km', 0),
        ml_score=ml_data.get('score', 0),
        shap_top3=', '.join(ml_data.get('top_factors', [])),
        retrieved_context='\\n\\n'.join([f"- {d['content']}" for d in retrieved_docs]),
    )
    
    response = _client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=1200,
        temperature=0.7,
    )
    return response.choices[0].message.content
'''

(ROOT / 'src' / 'rag' / 'generate.py').write_text(generate_py)
print('✓ Saved src/rag/generate.py')

## Summary

```
✅ Supabase tabel sederhana (no vector)
✅ 10 curated docs dengan topic tags
✅ Rule-based topic selection dari ML feature
✅ Metadata filter retrieval (Supabase native)
✅ Groq Llama 3.3 70B generation
✅ End-to-end test 3 lokasi
✅ Module saved: src/rag/retrieve.py + generate.py
```

**Resource usage**: 
- No embedding model loaded (no RAM hit)
- No vector search overhead
- 1 Supabase query + 1 Groq API call per analysis

**Next**: Build Streamlit UI yang panggil:
1. `extract_features(lat, lng)` → ML score
2. `retrieve_context(ml_data)` → relevant docs
3. `generate_summary(...)` → AI narrative